# Triton Matmul — Colab Runner (thin)

This notebook contains **no** kernel, timing, or correctness logic. It only sets up the repo on the import path and calls into the repository modules, so the exact same code runs here as anywhere else.

**Phase 0 only:** environment verification + a 256x256 smoke test. Run the cells top to bottom and paste the output back into Kiro before proceeding.

## Cell 1 — Environment verification
Confirms the runtime matches the confirmed stack (Tesla T4, CC 7.5, PyTorch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0). Do **not** reinstall or downgrade torch/triton.

In [ ]:
import torch, triton
print('torch        :', torch.__version__)
print('triton       :', triton.__version__)
print('cuda (torch) :', torch.version.cuda)
print('cuda avail   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu          :', torch.cuda.get_device_name(0))
    cc = torch.cuda.get_device_capability(0)
    print('compute cap  :', f'{cc[0]}.{cc[1]}')

## Cell 2 — Repo setup (pick ONE option)
Get the repository onto Colab's import path. Use **either** the git clone option **or** the upload/Drive option, then `sys.path.append` the repo root.

In [ ]:
# ---- Option A: clone from GitHub ------------------------------------------
# !git clone https://github.com/<your-user>/<your-repo>.git
# import sys; sys.path.append('/content/<your-repo>')

# ---- Option B: upload the folder or mount Google Drive --------------------
# from google.colab import drive; drive.mount('/content/drive')
# import sys; sys.path.append('/content/drive/MyDrive/<path-to-repo>')

# After choosing one, verify the modules import cleanly:
import sys
from kernels.matmul import MatmulConfig, matmul
print('kernels.matmul imported OK')
print('repo on path :', any('kernels' in p or 'content' in p for p in sys.path))

## Cell 3 — Phase 0 smoke test (256x256)
Runs the custom kernel with the conservative config (BLOCK_M=64, BLOCK_N=64, BLOCK_K=32, num_warps=4, num_stages=2) and compares to `torch.matmul`. The error is computed in FP32. If `tl.dot` is unsupported on this runtime, this cell will raise — **paste the full traceback** so we can diagnose the actual error.

In [ ]:
from tests.test_matmul import smoke_test_256
out = smoke_test_256()
print('Smoke test 256x256')
print('  output shape         :', out['shape'])
print('  output dtype         :', out['dtype'])
print('  config (BM,BN,BK,W,S):', out['config'])
print('  max abs error (FP32) : %.6e' % out['max_abs_error_fp32eval'])
print('  robust rel error     : %.6e' % out['robust_rel_error_fp32eval'])